# 04 · How often was each work printed? Editions in Spanish, 1500–1800

Source: the catalogue of early printed books of the Biblioteca Nacional de España (BNE),
limited to books in Spanish printed 1501–1800 (80,401 records). The catalogue is not included here:
set `CATALOG` and `PLACES` to your copy of it.

Rules (an approximation, not a bibliography):

- a record is an edition of a work if its title names the work (title pattern) and, when one is given,
  the author matches (author pattern); translations into Spanish count;
- collected works of an author ("Obras...", "Libros de la madre Teresa...") count as an edition of every
  work of that author in the corpus: the works circulated inside them;
- records with the same year and place count once (volumes or copies of one edition).

Output (in `DATA_DIR`): `editions.csv` (work_id, editions, records) and `editions_records.csv`
(every matched record, for checking).

## 1. Settings

In [1]:
DATA_DIR = "../data/results"                     # folder with the data files (see the README)
CATALOG = "../data/bne/catalog.tsv"              # BNE records in Spanish, 1501–1800 (columns: id, source, title, author_text, year)
PLACES = "../data/bne/bne_classified.tsv"        # place of printing of each BNE record (columns: id, lugar_de_publicacion)

In [2]:
import re, unicodedata
import pandas as pd


def norm(text):
    """Lowercase, no accents, old spellings folded. Example: 'Exercicios spirituales' -> 'exercicios spirituales',
    'Ejercicios' -> 'exercicios', 'Vida' -> 'uida', 'Christiana' -> 'cristiana', 'Theología' -> 'teologia'."""
    text = unicodedata.normalize("NFKD", str(text).lower())
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    for old, new in [("chr", "cr"), ("th", "t"), ("ph", "f"), ("v", "u"), ("j", "x"), ("y", "i"), ("ss", "s")]:
        text = text.replace(old, new)
    return text

### One rule per work: (title pattern, author pattern or None). Patterns are written in normal

In [3]:
# Spanish and folded with norm() like the catalogue, so "vida" also finds "uida" and "ejercicio" finds "exercicio".
RULES = {
    "agreda-cartas-felipe-iv": ("cartas", "agreda"),
    "agreda-mistica-ciudad": ("mistica ciudad", "agreda"),
    "agustin-confesiones": ("confesiones", r"agustin,+ +santo|agustin, santo|augustinus"),
    "almeida-tesoro-paciencia": ("tesoro de paciencia", "almeida"),
    "andrade-meditaciones-diarias": ("meditaciones diarias", r"andrade, alonso"),
    "angeles-dialogos": ("conquista del (reino|reyno)", "juan de los angeles"),
    "angeles-lucha": ("lucha espiritual", "juan de los angeles"),
    "angeles-manual": ("manual de vida perfecta", "juan de los angeles"),
    "angeles-presencia-dios": ("dios delante|presencia de dios", "juan de los angeles"),
    "angeles-soberanos-misterios": ("misterios de la misa", "juan de los angeles"),
    "angeles-triunfos": ("triunfos del amor", "juan de los angeles"),
    "angeles-vergel": ("vergel", "juan de los angeles"),
    "avila-audi-filia": (r"audi,? filia", r"juan de avila|avila, juan de"),
    "avila-avisos-reglas": ("avisos y reglas", r"juan de avila|avila, juan de"),
    "avila-cartas-ineditas": (r"\bcartas\b|epistolario", r"juan de avila|avila, juan de"),
    "avila-doctrina-mancebo": ("mancebo", r"juan de avila|avila, juan de"),
    "avila-epistolario": (r"epistolario|\bcartas\b", r"juan de avila|avila, juan de"),
    "avila-platicas": ("platicas", r"juan de avila|avila, juan de"),
    "avila-tratados-encarnacion": ("encarnacion", r"juan de avila|avila, juan de"),
    "avila-tratados-espiritu": ("espiritu santo", r"juan de avila|avila, juan de"),
    "avila-tratados-sacramento": ("sacramento", r"juan de avila|avila, juan de"),
    "belarmino-declaracion": ("declaracion copiosa", None),
    "cabrera-adviento": ("adviento", r"cabrera, alonso"),
    "cabrera-cuaresma": ("cuaresma", r"cabrera, alonso"),
    "cabrera-honras-felipe-ii": ("honras", r"cabrera, alonso"),
    "cano-victoria-de-si-mismo": ("victoria de si mismo", None),
    "carvajal-cortes-muerte": ("cortes de la muerte", None),
    "cepari-vida-gonzaga": ("gonzaga", "cepari"),
    "cisneros-ejercitatorio": ("ejercitatorio", None),
    "climaco-escala": ("escala", "climaco"),
    "crisostomo-sacerdocio": ("sacerdocio", r"juan crisostomo"),
    "cruz-cantico": ("cantico", r"juan de la cruz,+ +santo"),
    "cruz-cartas": (r"\bcartas\b", r"juan de la cruz,+ +santo"),
    "cruz-cautelas-avisos": ("cautelas", r"juan de la cruz,+ +santo"),
    "cruz-llama": ("llama de amor", r"juan de la cruz,+ +santo"),
    "cruz-noche": (r"noche (oscura|escura)", r"juan de la cruz,+ +santo"),
    "cruz-poesias-consejos": ("poesias|canciones", r"juan de la cruz,+ +santo"),
    "cruz-subida": ("subida del monte", r"juan de la cruz,+ +santo"),
    "estella-meditaciones": ("meditaciones", "diego de estella"),
    "figuera-suma": ("suma espiritual", "figuera"),
    "fleury-catecismo": ("catecismo historico", None),
    "godinez-teologia-mistica": ("teologia mistica", r"godinez, miguel|wadding"),
    "gracian-caminos-perfeccion": ("caminos de la perfeccion", None),
    "granada-breve-memorial": ("breve memorial", "luis de granada"),
    "granada-carta-euquerio": ("euquerio|eucherio", None),
    "granada-compendio-doctrina-cristiana": ("compendio (y explicacion )?de (la )?doctrina cristiana", "luis de granada"),
    "granada-compendio-doctrina-espiritual": ("compendio de (la )?doctrina espiritual", "luis de granada"),
    "granada-guia-pecadores": ("guia de pecadores", "luis de granada"),
    "granada-memorial": ("memorial de la vida cristiana", "luis de granada"),
    "granada-oracion": ("libro de la oracion|oracion y meditacion", "luis de granada"),
    "granada-retorica": ("retorica", "luis de granada"),
    "granada-simbolo": ("simbolo de la fe", "luis de granada"),
    "granada-trece-sermones": ("trece sermones", "luis de granada"),
    "granada-vida-avila": ("juan de avila", "luis de granada"),
    "granada-vida-bartolome": ("bartolome de los martires", None),
    "guevara-epistolas": ("epistolas familiares", None),
    "hernandez-alma-victoriosa": ("alma victoriosa", None),
    "isla-cartas-encina": ("juan de la encina", "isla, jose"),
    "isla-cartas-familiares": ("cartas familiares", "isla, jose"),
    "isla-dia-grande": ("dia grande de navarra", None),
    "isla-fray-gerundio": ("fray gerundio", None),
    "kempis-contemptus-mundi-granada": ("contemptus mundi|imitacion de cristo|imitacion de jesu", None),
    "leon-cantares": ("cantar de los cantares|cantares de salomon", r"leon, luis de"),
    "leon-job": (r"\bjob\b", r"leon, luis de"),
    "leon-nombres-cristo": ("nombres de cristo", None),
    "leon-perfecta-casada": ("perfecta casada", None),
    "leon-poesias": ("obras propias|poesias", r"leon, luis de"),
    "loyola-ejercicios": ("ejercicios espirituales", r"ignacio de loyola"),
    "madrid-arte-servir": ("arte para servir a dios", None),
    "madrid-espejo": ("espejo de ilustres personas", None),
    "malon-conversion-magdalena": ("magdalena", "malon"),
    "montesino-cancionero": ("cancionero", "montesino, ambrosio|ambrosio montesino"),
    "munoz-vida-avila": ("avila", r"munoz, luis"),
    "nieremberg-aficion-maria": ("aficion y amor de maria", None),
    "nieremberg-aprecio-gracia": ("aprecio y estima", None),
    "nieremberg-avisos": ("avisos espirituales", "nieremberg"),
    "nieremberg-catecismo": ("catecismo", "nieremberg"),
    "nieremberg-diferencia": ("diferencia entre lo temporal", None),
    "nieremberg-flores": ("flores espirituales", "nieremberg"),
    "nieremberg-vida-borja": ("borja", "nieremberg"),
    "nieremberg-vida-teresa": ("teresa", "nieremberg"),
    "osuna-tercer-abecedario": ("tercer abecedario|tercera parte del libro llamado abecedario", None),
    "palma-camino": ("camino espiritual", r"palma, luis de la"),
    "palma-pasion": ("pasion", r"palma, luis de la"),
    "parra-luz-verdades": ("luz de verdades", None),
    "pinamonti-director": ("director", "pinamonti"),
    "pinamonti-novisimos": ("novisimos", "pinamonti"),
    "puente-enfermedades": ("enfermedades", r"puente, luis de la"),
    "puente-meditaciones": ("meditaciones", r"puente, luis de la"),
    "puente-sentimientos": ("sentimientos", r"puente, luis de la"),
    "puente-vida-alvarez": ("baltasar alvarez", None),
    "ribadeneira-cisma": ("cisma", "ribadeneira|ribadeneyra"),
    "ribadeneira-principe": ("principe cristiano", "ribadeneira|ribadeneyra"),
    "ribadeneira-tribulacion": ("tribulacion", "ribadeneira|ribadeneyra"),
    "ribadeneira-vida-ignacio": ("ignacio", "ribadeneira|ribadeneyra"),
    "ribadeneira-vida-lainez": ("lainez", "ribadeneira|ribadeneyra"),
    "ribera-vida-teresa": ("teresa", r"ribera, francisco de, \( 1536"),
    "ripalda-catecismo": ("catecismo|doctrina cristiana", "ripalda"),
    "rodriguez-ejercicio": ("ejercicio de perfeccion", None),
    "romancero-sagrado": (None, None),                        # a 19th-century compilation: no early editions
    "rosignoli-verdades": ("verdades eternas", None),
    "sales-introduccion-silva": ("introduccion a la vida devota|filotea", None),
    "samaniego-prologo-galeato": ("prologo galeato", None),
    "samaniego-relacion-vida": ("relacion de la vida", "samaniego"),
    "segneri-cristiano": ("cristiano instruido", None),
    "talavera-breve-forma": ("breve forma de confesar", None),
    "teresa-atribuidos": (None, None),                          # a modern grouping of attributed texts
    "teresa-camino": ("camino de perfeccion", r"teresa de jesus,+ +santa"),
    "teresa-cartas": (r"\bcartas\b|epistolario", r"teresa de jesus,+ +santa"),
    "teresa-conceptos": ("conceptos del amor", r"teresa de jesus,+ +santa"),
    "teresa-constituciones": ("constituciones", r"teresa de jesus,+ +santa"),
    "teresa-exclamaciones": ("exclamaciones", r"teresa de jesus,+ +santa"),
    "teresa-fundaciones": ("fundaciones", r"teresa de jesus,+ +santa"),
    "teresa-modo-visitar": ("modo de visitar", r"teresa de jesus,+ +santa"),
    "teresa-moradas": ("moradas|castillo interior", r"teresa de jesus,+ +santa"),
    "teresa-relaciones": (r"\brelaciones\b", r"teresa de jesus,+ +santa"),
    "teresa-vida": (r"su vida|vida de la (santa )?madre teresa", r"teresa de jesus,+ +santa"),
    "torrubia-practica-ejercicios": ("practica de los ejercicios", None),
    "varones-ilustres": ("varones ilustres de la compania", None),
    "vegas-poesia-cristiana": ("poesia cristiana", None),
    "venegas-agonia": ("agonia del transito", None),
    "yepes-vida-teresa": ("teresa", "diego de yepes"),
    "zarate-discursos-paciencia": ("paciencia", r"zarate, hernando"),
}
# Collected works: (title pattern, author pattern) -> counts for every corpus work of that author (by work_id prefix)
COLLECTED = {
    "avila-": (r"^\W*(las |todas las |primero |segundo |tomo \w+ de las )?obras\b", r"juan de avila|avila, juan de"),
    "cruz-": (r"^\W*(las )?obras\b", r"juan de la cruz,+ +santo"),
    "granada-": (r"^\W*(las |todas las |primero |segundo |tomo \w+ de las )?obras\b", "luis de granada"),
    "isla-": (r"^\W*(las )?obras\b", "isla, jose"),
    "leon-": (r"^\W*(las )?obras\b", r"leon, luis de"),
    "nieremberg-": (r"^\W*(las )?obras\b", "nieremberg"),
    "teresa-": (r"^\W*(las |los )?(obras|libros de la)\b", r"teresa de jesus,+ +santa"),
    "angeles-": (r"^\W*(las )?obras\b", "juan de los angeles"),
}

### Catalogue

In [4]:
catalog = pd.read_csv(CATALOG, sep="\t", dtype=str)
bne = catalog[catalog["source"] == "bne"].copy()
places = pd.read_csv(PLACES, sep="\t", dtype=str, usecols=["id", "lugar_de_publicacion"])
bne = bne.merge(places, on="id", how="left")
bne["t"] = bne["title"].fillna("").map(norm)
bne["a"] = bne["author_text"].fillna("").map(norm)
bne["place"] = bne["lugar_de_publicacion"].fillna("").map(norm).str.replace(r"^(en|in|impresso en|impreso en) ", "", regex=True).str[:12]
print("BNE records:", len(bne))

def matches(title_pattern, author_pattern):
    hit = bne["t"].str.contains(norm(title_pattern), regex=True)
    if author_pattern:
        hit &= bne["a"].str.contains(norm(author_pattern), regex=True)
    return bne[hit]

BNE records: 80401


### Match by author: every BNE record of the author goes to the corpus work whose title it resembles most

In [5]:
# (cosine similarity of character 3–5-grams; e.g. "Libro llamado Camino de perfeccion" ~ "Camino de perfeccion").
# Below SIMILAR it is another work of that author and is left out. Works with no author pattern use their title pattern.
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings("ignore", "This pattern is interpreted as a regular expression")
SIMILAR = 0.30

works = pd.read_csv(f"{DATA_DIR}/texts.csv").groupby("work_id")["title"].first()
found = []
by_author = {}
for work_id in works.index:
    title_pattern, author_pattern = RULES[work_id]
    if author_pattern:
        by_author.setdefault(author_pattern, []).append(work_id)
    elif title_pattern:
        found.append(matches(title_pattern, None).assign(work_id=work_id, kind="title", similarity=1.0))

for author_pattern, work_ids in by_author.items():
    pool = bne[bne["a"].str.contains(norm(author_pattern), regex=True)].copy()
    if pool.empty:
        continue
    short_titles = pool["t"].str[:150]
    work_titles = [norm(works[w]) for w in work_ids]
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(list(short_titles) + work_titles)
    similarity = cosine_similarity(vectorizer.transform(short_titles), vectorizer.transform(work_titles))
    pool["work_id"] = [work_ids[i] for i in similarity.argmax(axis=1)]
    pool["similarity"] = similarity.max(axis=1)
    pool["kind"] = "title"
    # The title pattern also counts: long titles can resemble the short corpus title too little
    # (e.g. Yepes, "Vida, virtudes y milagros de la bienaventurada virgen Teresa de Jesus")
    by_pattern = pd.Series(False, index=pool.index)
    for w in work_ids:
        hit = pool["t"].str.contains(norm(RULES[w][0]), regex=True) & ~by_pattern
        pool.loc[hit, "work_id"] = w
        by_pattern |= hit
    found.append(pool[(pool["similarity"] >= SIMILAR) | by_pattern])
    # Collected works of this author count for each of its corpus works
    prefix = work_ids[0].split("-")[0] + "-"
    if prefix in COLLECTED:
        t, a = COLLECTED[prefix]
        collected = pool[pool["t"].str.contains(norm(t), regex=True)]
        for w in work_ids:
            found.append(collected.assign(work_id=w, kind="collected", similarity=1.0))

records = pd.concat(found).sort_values("kind", ascending=False).drop_duplicates(["work_id", "id"])
editions = (records.drop_duplicates(["work_id", "year", "place"]).groupby("work_id").size()
            .reindex(works.index, fill_value=0).rename("editions"))
# Varones ilustres is a series of volumes printed once each (1643-1647, 1666-1667, 1734-1736), not reprints
OVERRIDES = {"varones-ilustres": 1}
for work_id, n in OVERRIDES.items():
    editions[work_id] = n
separate = (records[records["kind"] == "title"].drop_duplicates(["work_id", "year", "place"]).groupby("work_id").size()
            .reindex(works.index, fill_value=0))
table = pd.DataFrame({"title": works, "editions": editions, "separate": separate,
                      "records": records.groupby("work_id").size().reindex(works.index, fill_value=0)})
print(table["editions"].describe().round(1))
print(table.sort_values("editions", ascending=False).to_string())

count    123.0
mean       9.6
std        9.7
min        0.0
25%        3.0
50%        6.0
75%       14.0
max       38.0
Name: editions, dtype: float64
                                                                                         title  editions  separate  records
work_id                                                                                                                    
kempis-contemptus-mundi-granada                                               Contemptus mundi        38        38       40
teresa-cartas                                                                           Cartas        36        14       49
granada-oracion                                               Libro de la oración y meditación        36        26       71
granada-memorial                                                 Memorial de la vida cristiana        32        21       67
teresa-conceptos                                                    Conceptos del amor de Dios        30 

### Save

In [6]:
table[["editions", "separate", "records"]].to_csv(f"{DATA_DIR}/editions.csv")
records[["work_id", "kind", "similarity", "id", "year", "place", "title", "author_text"]].to_csv(f"{DATA_DIR}/editions_records.csv", index=False)